In [1]:
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d
import yaml

# ---------------------------------
# Display settings 
# ---------------------------------
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# ------------------------------
# Load configuration
# ------------------------------
with open('../../Settings.yaml', 'r') as file:
    Setting = yaml.safe_load(file)

# ------------------------------
# Load raw dataset from Excel
# ------------------------------
file_name = 'Blackout.xlsx'
file_path = f"{Setting['Raw_Path']}/{file_name}"
df = pd.read_excel(file_path)

# ------------------------------
# Column name to be cleaned and interpolated/extrapolated
# ------------------------------
col = 'Total Section Not Distributed Electricity (Milion KWH)'

# ------------------------------
# Ensure Year is numeric (int)
# ------------------------------
df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype(int)

# ------------------------------
# Ensure the target column is numeric
# (non-numeric values become NaN)
# ------------------------------
df[col] = pd.to_numeric(df[col], errors='coerce')

# ------------------------------
# Build a complete yearly series from 1381 to 1401
# Missing years become NaN
# ------------------------------
s = (df.set_index('Year')[col]
       .sort_index()
       .reindex(range(1381, 1402)))

# ------------------------------
# Log-linear extrapolation/interpolation:
# Fit a linear function on log(values) over available years,
# then exponentiate back to get positive filled values.
# This fills:
#   - internal gaps (interpolation)
#   - initial missing years before first observation (extrapolation)
#   - and would also fill after last observation if needed
# ------------------------------
x = s.dropna().index.to_numpy(dtype=float)                 
y_log = np.log(s.dropna().to_numpy(dtype=float))           

# Create interpolation function on log scale (with extrapolation enabled)
f_log = interp1d(
    x, y_log,
    kind='linear',
    fill_value='extrapolate',
    bounds_error=False
)

# Evaluate for all years in the full index and convert back from log
s_filled = pd.Series(
    np.exp(f_log(s.index.to_numpy(dtype=float))),
    index=s.index
)

# ------------------------------
# Convert the filled series into a DataFrame
# ------------------------------
df_final = s_filled.reset_index()
df_final.columns = ['Year', col]


# ------------------------------
# Merge filled values back into the original dataframe on Year
# (drop the original col first to avoid duplicates)
# ------------------------------
Blackout = (df.drop(columns=[col], errors='ignore')
              .merge(df_final, on='Year', how='left')
              .sort_values('Year')
              .reset_index(drop=True))


Blackout['Blackout'] = (
    Blackout['Total Section Not Distributed Electricity (Milion KWH)']
    / Blackout['Industrial Electricity Consumption (Milion KWH)']
)


# ------------------------------
# Export Blackout by year 
# ------------------------------
output_file_name = 'Unadjusted.xlsx'
sheet_name= 'Blackout'
path = f"{Setting['Output_Path_Unajusted']}/{output_file_name}"
with pd.ExcelWriter(path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    Blackout.to_excel(writer,sheet_name=sheet_name ,index=False)
